<a href="https://colab.research.google.com/github/savitoj370/CRUD/blob/main/summarize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U \
transformers==4.51.3 \
accelerate \
qwen-vl-utils \
sentencepiece \
einops

In [2]:
import transformers
print(transformers.__version__)

4.51.3


In [3]:
import os
import gc
import cv2
import torch

from PIL import Image

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM
)

from qwen_vl_utils import process_vision_info

In [4]:
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen2_5_VLForConditionalGeneration(
  (visual): Qwen2_5_VisionTransformerPretrainedModel(
    (patch_embed): Qwen2_5_VisionPatchEmbed(
      (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
    )
    (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-31): 32 x Qwen2_5_VLVisionBlock(
        (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
        (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
        (attn): Qwen2_5_VLVisionSdpaAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (mlp): Qwen2_5_VLMLP(
          (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
          (act_fn): SiLU()
        )
      )
    )
    (merger): Qwen2_5_VLPatchMerger

In [5]:
!ffmpeg -y -i input_brown.mp4 -c:v libx264 -pix_fmt yuv420p -c:a aac converted.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [6]:
import os
import cv2
video_path = "converted.mp4"

output_folder = "output_frames"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)
#print(os.path.exists(video_path))
print(video_path)
video = cv2.VideoCapture(video_path)
print("Frame Count:", int(video.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", video.get(cv2.CAP_PROP_FPS))
print("Duration:", video.get(cv2.CAP_PROP_FRAME_COUNT) / video.get(cv2.CAP_PROP_FPS))
if not video.isOpened():
    print("Error: Unable to open video file.")
    exit()
frame_number=0
while True:
    success, frame = video.read()

    if not success:
        break

    frame_name = os.path.join(
        output_folder,
        f"frame_{frame_number:05d}.jpg"
    )

    cv2.imwrite(frame_name, frame)

    print(f"Saved {frame_name}")

    frame_number += 1

video.release()

print("\nExtraction completed!")
print(f"Total frames extracted: {frame_number}")

converted.mp4
Frame Count: 482
FPS: 23.976023976023978
Duration: 20.103416666666664
Saved output_frames/frame_00000.jpg
Saved output_frames/frame_00001.jpg
Saved output_frames/frame_00002.jpg
Saved output_frames/frame_00003.jpg
Saved output_frames/frame_00004.jpg
Saved output_frames/frame_00005.jpg
Saved output_frames/frame_00006.jpg
Saved output_frames/frame_00007.jpg
Saved output_frames/frame_00008.jpg
Saved output_frames/frame_00009.jpg
Saved output_frames/frame_00010.jpg
Saved output_frames/frame_00011.jpg
Saved output_frames/frame_00012.jpg
Saved output_frames/frame_00013.jpg
Saved output_frames/frame_00014.jpg
Saved output_frames/frame_00015.jpg
Saved output_frames/frame_00016.jpg
Saved output_frames/frame_00017.jpg
Saved output_frames/frame_00018.jpg
Saved output_frames/frame_00019.jpg
Saved output_frames/frame_00020.jpg
Saved output_frames/frame_00021.jpg
Saved output_frames/frame_00022.jpg
Saved output_frames/frame_00023.jpg
Saved output_frames/frame_00024.jpg
Saved output_fra

In [8]:
import os
import gc
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info

frame_folder = "output_frames"

frame_files = sorted(
    [f for f in os.listdir(frame_folder) if f.endswith(".jpg")]
)

frame_descriptions = []

for idx, frame_file in enumerate(frame_files):

    print(f"\nProcessing {idx+1}/{len(frame_files)} : {frame_file}")

    try:
        image_path = os.path.join(frame_folder, frame_file)

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image_path
                    },
                    {
                        "type": "text",
                        "text": """
Analyze this frame.

Ignore all text, subtitles, captions,
labels, logos, numbers and watermarks.

Provide:

Objects:
- Important objects visible

People:
- Number of people and appearance

Actions:
- What is happening

Scene:
- Environment or location

Relationships:
- Interactions between people and objects

Summary:
- A short 2-3 sentence summary describing this frame.

Do NOT perform OCR.
"""
                    }
                ]
            }
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        image_inputs, video_inputs = process_vision_info(messages)
        print("Step 1: Preparing inputs")
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        print("Step 2: Moving to GPU")
        inputs = inputs.to(model.device)
        print("Step 3: Starting generation")
        with torch.no_grad():

            generated_ids = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False
            )
        print("Step 4: Generation finished")
        generated_ids = [
            out[len(inp):]
            for inp, out in zip(inputs.input_ids, generated_ids)
        ]

        response = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )[0]

        frame_descriptions.append(response)

        print("Done")

        if (idx + 1) % 20 == 0:

            with open("frame_descriptions.txt", "w", encoding="utf-8") as f:

                for i, desc in enumerate(frame_descriptions):

                    f.write(f"Frame {i+1}\n")
                    f.write(desc)
                    f.write("\n")
                    f.write("=" * 80)
                    f.write("\n")

            print(f"Progress saved at frame {idx+1}")

    except Exception as e:

        print(f"Error processing {frame_file}")
        print(e)

    finally:

        if 'inputs' in locals():
            del inputs

        if 'generated_ids' in locals():
            del generated_ids

        if 'image_inputs' in locals():
            del image_inputs

        if 'video_inputs' in locals():
            del video_inputs

        gc.collect()

        if torch.cuda.is_available() and torch.cuda.is_initialized():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
print("\nFinished generating descriptions.")


Processing 1/482 : frame_00000.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting generation


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Step 4: Generation finished
Done

Processing 2/482 : frame_00001.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting generation
Step 4: Generation finished
Done

Processing 3/482 : frame_00002.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting generation
Step 4: Generation finished
Done

Processing 4/482 : frame_00003.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting generation
Step 4: Generation finished
Done

Processing 5/482 : frame_00004.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting generation
Step 4: Generation finished
Done

Processing 6/482 : frame_00005.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting generation
Step 4: Generation finished
Done

Processing 7/482 : frame_00006.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting generation
Step 4: Generation finished
Done

Processing 8/482 : frame_00007.jpg
Step 1: Preparing inputs
Step 2: Moving to GPU
Step 3: Starting gen

In [9]:
print(len(frame_descriptions))

482


In [10]:
with open("frame_descriptions.txt", "r", encoding="utf-8") as f:
    frame_text = f.read()

print(frame_text[:2000])

Frame 1
**Objects:**
- A whiteboard with some writing on it.
- A person's face in the foreground, partially out of focus.

**People:**
- One person is visible in the foreground, appearing to be speaking or shouting.

**Actions:**
- The person in the foreground seems to be engaged in an intense conversation or argument, as indicated by their open mouth and expressive facial features.

**Scene:**
- The setting appears to be an indoor environment, possibly a classroom or meeting room, given the presence of the whiteboard.

**Relationships:**
- The person in the foreground is likely addressing the audience or another individual not visible in the frame.

**Summary:**
- A person is passionately speaking or shouting in front of a
Frame 2
**Objects:**
- A whiteboard with some writing on it.
- A person's face in the foreground, partially out of focus.

**People:**
- One person is visible in the foreground, appearing to be speaking or shouting.

**Actions:**
- The person in the foreground seems

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

text_model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(text_model_name)

text_model = AutoModelForCausalLM.from_pretrained(
    text_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

text_model.eval()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (rotary_emb):

In [12]:
prompt = f"""
You are an AI assistant specialized in video understanding.

The following text consists of descriptions generated from consecutive
frames extracted from the same video.

Each description represents one moment in time.

Your task is to reconstruct the overall story of the video.

Instructions:

• Read all frame descriptions carefully.
• Combine repeated observations.
• Preserve chronological order.
• Track how people move.
• Track how objects move.
• Describe how actions change over time.
• Mention scene transitions if any.
• Focus on the main event.
• Produce ONE coherent summary.
• Do NOT mention frame numbers.
• Do NOT repeat information.
• Assume all descriptions belong to the same continuous video.

Frame Descriptions:

{frame_text}

Generate a detailed summary of the entire video.
"""

In [13]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'converted.mp4', 'input_brown.mp4', 'frame_descriptions.txt', 'output_frames', 'sample_data']


In [14]:
import os

for root, dirs, files in os.walk("."):
    if "frame_descriptions.txt" in files:
        print(os.path.join(root, "frame_descriptions.txt"))

./frame_descriptions.txt


In [15]:
with open("frame_descriptions.txt", "r", encoding="utf-8") as f:
    frame_text = f.read()

frames = frame_text.split("=" * 80)

frames = [f.strip() for f in frames if f.strip()]

print("Total frame descriptions:", len(frames))

chunk_size = 20

chunks = [
    frames[i:i+chunk_size]
    for i in range(0, len(frames), chunk_size)
]

print("Total chunks:", len(chunks))

Total frame descriptions: 480
Total chunks: 24


In [16]:
chunk_summaries = []

for idx, chunk in enumerate(chunks):

    print(f"Summarizing Chunk {idx+1}/{len(chunks)}")

    chunk_text = "\n\n".join(chunk)

    prompt = f"""
You are an AI assistant specialized in video understanding.

The following text contains descriptions generated from consecutive frames.

Generate ONE concise summary describing what happens during this part of the video.

Instructions:
- Merge repeated information.
- Preserve chronological order.
- Mention important actions.
- Mention scene changes.
- Do not mention frame numbers.

Frame Descriptions:

{chunk_text}

Summary:
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]

    summary = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    chunk_summaries.append(summary)

    print(summary)
    print("-"*80)

    del inputs
    del generated_ids
    torch.cuda.empty_cache()

Summarizing Chunk 1/24
A person is passionately speaking or shouting in front of a whiteboard, then transitions to an outdoor setting where they smile and look at the camera while holding a smartphone.
--------------------------------------------------------------------------------
Summarizing Chunk 2/24
A person in a blue jumpsuit and yellow boots walks on a dirt road, holding a white cloth in their left hand, passing by large trucks in an industrial or commercial area. The scene shifts slightly, showing another person in a similar jumpsuit near one of the trucks. The environment remains consistent with trees and open sky in the distance, suggesting a work-related setting.
--------------------------------------------------------------------------------
Summarizing Chunk 3/24
A person in a blue jumpsuit and yellow boots walks on a dirt road in an industrial setting, holding a piece of clothing. They pass by large trucks and another worker near one of them, suggesting a work-related act

In [17]:
with open("chunk_summaries.txt", "w", encoding="utf-8") as f:

    for i, summary in enumerate(chunk_summaries):

        f.write(f"Chunk {i+1}\n")
        f.write(summary)
        f.write("\n")
        f.write("="*80)
        f.write("\n")

print("Chunk summaries saved.")

Chunk summaries saved.


In [18]:
all_chunks = "\n\n".join(chunk_summaries)

prompt = f"""
You are an AI assistant specialized in video understanding.

The following text contains summaries from different chronological parts
of the same video.

Your task is to generate ONE complete summary of the entire video.

Instructions:
- Preserve chronological order.
- Merge repeated information.
- Describe how the events evolve.
- Mention scene transitions.
- Produce one natural paragraph.
- Do not mention chunk numbers.

Chunk Summaries:

{all_chunks}

Final Video Summary:
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=[text],
    return_tensors="pt"
)

inputs = inputs.to(model.device)

with torch.no_grad():

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]

video_summary = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print(video_summary)

with open("video_summary.txt", "w", encoding="utf-8") as f:
    f.write(video_summary)

print("Final video summary saved.")

The video begins with a person passionately speaking or shouting in front of a whiteboard, setting a dynamic and intense tone. The scene then transitions to an outdoor setting where the same person smiles and looks at the camera while holding a smartphone, creating a more relaxed and personal atmosphere. The narrative shifts to a work-related environment, where a person in a blue jumpsuit and yellow boots walks on a dirt road, holding a white cloth, passing by large trucks and another worker near one of them. This scene emphasizes a busy and industrious setting, contrasting with the previous personal and intense moments. The video then moves to a serene outdoor path, where a person walks holding a camera, capturing the beauty of nature under a clear sky. This segment provides a peaceful and reflective contrast to the earlier industrial scenes. The video continues with a man engaged in a phone conversation at a counter in various settings, including a kitchen, a restaurant, and a fast-f